In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

In [3]:
customers = spark.read.format('csv').option('inferSchema', True).option('header', True).load('output/silver/customers')
order_items = spark.read.format('csv').option('inferSchema', True).option('header', True).load('output/silver/order_items')
orders = spark.read.format('csv').option('inferSchema', True).option('header', True).load('output/silver/orders')
payments = spark.read.format('csv').option('inferSchema', True).option('header', True).load('output/silver/payments')
products = spark.read.format('csv').option('inferSchema', True).option('header', True).load('output/silver/products')
sellers = spark.read.format('csv').option('inferSchema', True).option('header', True).load('output/silver/sellers')


In [4]:
cust_orders = customers.join(orders, "customer_id", "left")

In [5]:
from pyspark.sql import Window

In [6]:
from pyspark.sql.functions import *

In [7]:
window_spec = Window.partitionBy(
    "customer_unique_id"
).orderBy(
    col("order_purchase_timestamp"
).desc()
)

In [8]:
latest = cust_orders.withColumn(
    "rn", row_number().over(window_spec)
).filter("rn = 1")

In [10]:
agg = cust_orders.groupBy("customer_unique_id").agg(
    min("order_purchase_timestamp").alias("first_order_date"),
    countDistinct("order_id").alias("total_orders")
)

In [11]:
latest.show()

+--------------------+-----------------+--------------+--------------------+------------------------+--------------------+--------------------+---------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------------+---------+---+
|         customer_id|    customer_city|customer_state|  customer_unique_id|customer_zip_code_prefix|        _ingested_at|    _source_endpoint|_is_valid|            order_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|        _ingested_at|   _source_endpoint|_is_valid| rn|
+--------------------+-----------------+--------------+--------------------+------------------------+--------------------+--------------------+---------+--------------------+------------+------------------------+-------------------+--------------

In [12]:
agg.show()

+--------------------+-------------------+------------+
|  customer_unique_id|   first_order_date|total_orders|
+--------------------+-------------------+------------+
|000c8bdb58a29e711...|2017-12-12 22:53:35|           1|
|0e1a29e5ec2a8b40b...|2017-02-20 22:26:30|           1|
|38f7df2ba1b785cfb...|2018-04-03 11:59:08|           1|
|384ffc528dd30fd49...|2018-04-27 15:42:08|           1|
|958da6ff0122fa69b...|2017-08-16 15:34:09|           1|
|8f8eb77a8d3f39241...|2018-04-18 13:06:52|           1|
|72ac8059a9a54b1b8...|2017-08-17 15:24:09|           1|
|762efb8c186a318d9...|2017-06-02 18:12:44|           1|
|c2676cdedacba0e3a...|2017-06-01 16:10:28|           1|
|45ba2acc9e8cbf07b...|2018-06-10 21:20:12|           1|
|7462a753a77ed0933...|2018-08-11 21:20:06|           2|
|2e99a768d2ba4e4a4...|2017-07-23 21:11:24|           1|
|14a188558af6cd5bc...|2018-01-26 18:30:17|           2|
|35d0b790a53b0d57a...|2017-05-07 21:24:45|           1|
|14e7273c0d9ae09dd...|2018-04-03 11:05:02|      

In [14]:
spend = order_items.groupBy("order_id").agg(
    (sum("price") + sum("freight_value")).alias("order_total")
)

In [16]:
cust_spend = cust_orders.join(spend, "order_id", "left") \
    .groupBy("customer_unique_id") \
    .agg(sum("order_total").alias("total_spend"))

In [17]:
spend.show()

+--------------------+------------------+
|            order_id|       order_total|
+--------------------+------------------+
|0baa56401ae212e30...|             87.22|
|0bcc4cc85c24f23e2...|             195.0|
|0d4a43cf7b7994e0a...|37.379999999999995|
|16e233a1bc2e99095...| 82.00999999999999|
|240891986117bf8fd...|             38.12|
|251f0a3981c4a8cb8...| 93.14999999999999|
|2d08fef3d5af150cd...|            260.52|
|2f5e702c70321e3f1...| 83.17999999999999|
|360b84029c125bb8d...|             31.77|
|39a2c5163845188e0...|            176.78|
|3ba1f632cf89a08ca...|            190.62|
|3d44f7b0d44056fcb...|            118.17|
|3e22c141d28618608...|220.14000000000001|
|3ee6694bb99d5a6db...|             54.92|
|3f090361c1e3266bf...|             28.88|
|416699ef2e608b6f2...|            146.02|
|41d85a7a138b7205e...|            153.82|
|444a494f39275142d...|              67.5|
|46a1ac51daefad9fb...|            141.26|
|4b0e0e0eba7274a1e...|            158.54|
+--------------------+------------

In [18]:
cust_spend.show()

+--------------------+------------------+
|  customer_unique_id|       total_spend|
+--------------------+------------------+
|94c1c602b16a9fdfb...|            2566.9|
|d6a47604b551804c2...|            156.33|
|4c7c157b7ca1593ff...|200.45000000000002|
|38f7df2ba1b785cfb...|226.79000000000002|
|384ffc528dd30fd49...|             61.69|
|67903a0796622001d...|57.269999999999996|
|f2a9bc9a1db05c873...|            122.42|
|ddd9c384b4b7a2f80...|222.17000000000002|
|65be6248a39942f9a...|             64.91|
|2d6d1699603a346a2...|            173.13|
|7462a753a77ed0933...|            145.72|
|57cce8aaddd9bf6ac...|            350.34|
|f8ec2178c6a49d0c2...|             36.35|
|20f42618c40034d85...|            337.77|
|a99697757b492ec1b...|             306.2|
|3db49645cbf26782b...|            553.04|
|9022006230939835a...|             55.35|
|0d1fe68a782a79b00...|             68.35|
|7b0eaf68a16e4808e...|           2340.08|
|45ba2acc9e8cbf07b...|             73.37|
+--------------------+------------

In [20]:
dim_customers = latest.join(agg, "customer_unique_id") \
    .join(cust_spend, "customer_unique_id") \
    .select(
        monotonically_increasing_id().alias("customer_key"),
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "customer_zip_code_prefix",
        "first_order_date",
        "total_orders",
        "total_spend",
        (col("total_orders") > 1).alias("is_repeat_customer")
    )

In [21]:
dim_customers.show()

+------------+--------------------+--------------------+--------------+------------------------+-------------------+------------+------------------+------------------+
|customer_key|  customer_unique_id|       customer_city|customer_state|customer_zip_code_prefix|   first_order_date|total_orders|       total_spend|is_repeat_customer|
+------------+--------------------+--------------------+--------------+------------------------+-------------------+------------+------------------+------------------+
|           0|000c8bdb58a29e711...|      belo horizonte|            MG|                   31555|2017-12-12 22:53:35|           1|              29.0|             false|
|           1|0e1a29e5ec2a8b40b...|           sao paulo|            SP|                    4942|2017-02-20 22:26:30|           1|            126.37|             false|
|           2|38f7df2ba1b785cfb...|         carapicuiba|            SP|                    6325|2018-04-03 11:59:08|           1|226.79000000000002|            

In [24]:
dim_products = products.select(
    monotonically_increasing_id().alias("product_key"),
    "product_id",
    coalesce("product_category_name", lit("unknown")).alias("product_category_name"),
    "product_weight_g",
    (col("product_length_cm") * 
     col("product_height_cm") * 
     col("product_width_cm")).alias("product_volume_cm3"),
    "product_photos_qty",
    col("product_description_lenght").alias("product_description_length")
)

In [25]:
dim_sellers = sellers.select(
    monotonically_increasing_id().alias("seller_key"),
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix"
)

In [27]:
payment_total = payments.groupBy("order_id") \
    .agg(sum("payment_value").alias("total_payment"))

In [29]:
window_payment = Window.partitionBy("order_id").orderBy(
    col("payment_value").desc(),
    col("payment_type").asc()
)

payment_type = payments.withColumn("rn", row_number().over(window_payment)) \
    .filter("rn = 1") \
    .select("order_id", "payment_type")

In [30]:
payment_installments = payments.groupBy("order_id") \
    .agg(max("payment_installments").alias("payment_installments"))

In [32]:
# Step 1: Join order_items with orders
fact = order_items.join(orders, "order_id")

# Step 2: Bring customer_unique_id from customers
fact = fact.join(
    customers.select("customer_id", "customer_unique_id"),
    "customer_id",
    "left"
)

# Step 3: Now join with dim_customers
fact = fact.join(
    dim_customers,
    "customer_unique_id",
    "left"
)

# Step 4: Other joins
fact = fact.join(dim_products, "product_id", "left") \
           .join(dim_sellers, "seller_id", "left") \
           .join(payment_total, "order_id", "left") \
           .join(payment_type, "order_id", "left") \
           .join(payment_installments, "order_id", "left")

In [34]:
# total item price per order
order_price = order_items.groupBy("order_id") \
    .agg(sum("price").alias("total_price"))

fact = fact.join(order_price, "order_id")

fact = fact.withColumn(
    "payment_value",
    (col("price") / col("total_price")) * col("total_payment")
)

In [36]:
fact = fact.withColumn(
    "days_to_deliver",
    datediff("order_delivered_customer_date", "order_purchase_timestamp")
).withColumn(
    "days_delivery_vs_estimate",
    datediff("order_delivered_customer_date", "order_estimated_delivery_date")
).withColumn(
    "is_late_delivery",
    when(col("days_delivery_vs_estimate") > 0, True)
)

In [37]:
fact_order_items = fact.select(
    monotonically_increasing_id().alias("order_item_sk"),
    "order_id",
    "order_item_id",
    "customer_key",
    "product_key",
    "seller_key",
    to_date("order_purchase_timestamp").alias("order_date"),
    "order_status",
    "price",
    "freight_value",
    "payment_value",
    "payment_type",
    "payment_installments",
    "days_to_deliver",
    "days_delivery_vs_estimate",
    "is_late_delivery"
)

In [38]:
dim_customers.write.mode("overwrite").csv("output/gold/dim_customers", header=True)
dim_products.write.mode("overwrite").csv("output/gold/dim_products", header=True)
dim_sellers.write.mode("overwrite").csv("output/gold/dim_sellers", header=True)
fact_order_items.write.mode("overwrite").csv("output/gold/fact_order_items", header=True)